In [21]:
# =========================================
# 🧪 ADVANCED CLINICAL AI (PUBMED-STYLE + MULTI-AGENT)
# =========================================



In [2]:
!pip install -q langchain langchain-groq langchain-community tavily-python python-dotenv


In [5]:

import requests
from dotenv import load_dotenv
import os
load_dotenv("/voc/work/.env")

True

In [6]:

from langchain_groq import ChatGroq



llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

response = llm.invoke("Explain Agentic AI in 20 words")

print(response.content)

/usr/local/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Agentic AI refers to autonomous systems that exhibit self-awareness, goal-directed behavior, and decision-making capabilities like human agents.


In [7]:
# =========================================
# 🔬 STEP 1: PUBMED SEARCH (E-UTILS)
# =========================================
def pubmed_search(query, max_results=3):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    # Step 1: Search IDs
    search_url = f"{base}esearch.fcgi"
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json"
    }
    
    res = requests.get(search_url, params=params).json()
    ids = res["esearchresult"]["idlist"]
    
    if not ids:
        return "No PubMed results found"
    
    # Step 2: Fetch abstracts
    fetch_url = f"{base}efetch.fcgi"
    fetch_params = {
        "db": "pubmed",
        "id": ",".join(ids),
        "retmode": "text",
        "rettype": "abstract"
    }
    
    abstracts = requests.get(fetch_url, params=fetch_params).text
    return abstracts[:2000]  # limit size

In [17]:
def pubmed_search(query, max_results=3):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    try:
        # Step 1: Search IDs
        search_url = f"{base}esearch.fcgi"
        params = {
            "db": "pubmed",
            "term": query,
            "retmax": max_results,
            "retmode": "json"
        }
        
        res = requests.get(search_url, params=params)
        
        # 🔥 check response
        if res.status_code != 200:
            return "PubMed API error"
        
        data = res.json()
        
        # 🔥 SAFE ACCESS
        ids = data.get("esearchresult", {}).get("idlist", [])
        
        if not ids:
            return f"No PubMed results for: {query}"
        
        # Step 2: Fetch abstracts
        fetch_url = f"{base}efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ",".join(ids),
            "retmode": "text",
            "rettype": "abstract"
        }
        
        abstracts = requests.get(fetch_url, params=fetch_params).text
        
        return abstracts[:2000]
    
    except Exception as e:
        return f"PubMed Error: {str(e)}"

In [16]:
# =========================================
# 🧠 AGENT 1: QUERY DECOMPOSITION
# =========================================
def decompose(query):
    response = llm.invoke(f"""
Generate 3 high-quality PubMed search queries.

Rules:
- Use medical terms
- Keep concise
- Avoid numbering

Query: {query}
""").content
    
    queries = [q.strip("- ").strip() for q in response.split("\n") if q.strip()]
    
    return queries[:3]

In [14]:
# =========================================
# 🔍 AGENT 2: PUBMED RETRIEVAL
# =========================================
def retrieve_pubmed(queries):
    all_data = []
    for q in queries:
        data = pubmed_search(q)
        all_data.append(data)
    return "\n\n".join(all_data)


In [10]:
# =========================================
# 📚 AGENT 3: EVIDENCE SUMMARIZATION
# =========================================
def summarize_evidence(evidence):
    return llm.invoke(f"""
You are reviewing clinical literature.

Extract:
- key risks
- interaction findings
- important insights

Evidence:
{evidence}

Be concise and factual.
""").content


In [12]:
# =========================================
# 🧠 AGENT 4: CLINICAL REASONING
# =========================================
def clinical_reasoning(query, summary):
    return llm.invoke(f"""
You are a clinical pharmacology expert.

Patient Query:
{query}

Evidence Summary:
{summary}

Provide:
1. Interaction risk
2. Mechanism
3. Safer alternatives
4. Recommendation

Be medically responsible.
""").content

In [19]:
def validate(answer):
    return llm.invoke(f"""
You are a medical safety validator.

Check the answer for:
- hallucinations
- unsafe medical advice
- missing warnings

If needed, correct the answer.

Answer:
{answer}

Return improved version.
""").content

In [20]:
# =========================================
# 🔄 FULL PIPELINE
# =========================================
def pharma_pubmed_system(query):
    print("🧠 Decomposing query...")
    queries = decompose(query)
    
    print("🔬 Fetching PubMed abstracts...")
    evidence = retrieve_pubmed(queries)
    
    print("📚 Summarizing evidence...")
    summary = summarize_evidence(evidence)
    
    print("🧠 Clinical reasoning...")
    answer = clinical_reasoning(query, summary)
    
    print("🛡️ Validating...\n")
    return validate(answer)

# =========================================
# 🚀 TEST
# =========================================
print(pharma_pubmed_system(
    "Patient taking warfarin and ibuprofen. What are risks and safer alternatives?"
))

🧠 Decomposing query...
🔬 Fetching PubMed abstracts...
📚 Summarizing evidence...
🧠 Clinical reasoning...
🛡️ Validating...

**Patient Query: Warfarin and Ibuprofen Interaction**

**1. Interaction Risk:**
The combination of warfarin and ibuprofen increases the risk of gastrointestinal bleeding (GIB) and hematuria. Warfarin is an anticoagulant that inhibits vitamin K-dependent clotting factors, while ibuprofen is a nonsteroidal anti-inflammatory drug (NSAID) that can cause platelet dysfunction and gastrointestinal mucosal damage. This combination may lead to an increased risk of bleeding, particularly in patients with a history of gastrointestinal bleeding or those taking other medications that increase the risk of bleeding.

**2. Mechanism:**
The mechanism of interaction between warfarin and ibuprofen involves the inhibition of platelet aggregation and the disruption of the gastrointestinal mucosa. Ibuprofen can cause platelet dysfunction by inhibiting the production of thromboxane A2, a 

In [24]:
# =========================================
# 🧪 CLINICAL AI UI (PUBMED + LIVE THINKING)
# =========================================

# !pip install -q gradio requests langchain langchain-groq python-dotenv

import gradio as gr
import requests
import os
from dotenv import load_dotenv

# load_dotenv("work/.env")

from langchain_groq import ChatGroq
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# =========================================
# 🔬 PUBMED SEARCH (ROBUST)
# =========================================
def pubmed_search(query, max_results=2):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    try:
        search_url = f"{base}esearch.fcgi"
        params = {"db": "pubmed", "term": query, "retmax": max_results, "retmode": "json"}
        res = requests.get(search_url, params=params).json()
        
        ids = res.get("esearchresult", {}).get("idlist", [])
        if not ids:
            return "No strong PubMed evidence found."
        
        fetch_url = f"{base}efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ",".join(ids),
            "retmode": "text",
            "rettype": "abstract"
        }
        
        return requests.get(fetch_url, params=fetch_params).text[:1500]
    
    except:
        return "Error retrieving PubMed data"

# =========================================
# 🧠 STEP 1: DECOMPOSE
# =========================================
def decompose(query):
    return llm.invoke(f"""
Generate 3 PubMed search queries:

Query: {query}
""").content.split("\n")

# =========================================
# 🔄 MAIN PIPELINE (STREAMING STYLE)
# =========================================
def pharma_live(query):
    logs = []
    
    logs.append("🧠 Decomposing clinical query...")
    queries = decompose(query)
    
    logs.append("🔬 Searching PubMed...")
    evidence = ""
    for q in queries[:2]:
        evidence += pubmed_search(q) + "\n\n"
    
    logs.append("📚 Summarizing evidence...")
    summary = llm.invoke(f"Summarize key clinical findings:\n{evidence}").content
    
    logs.append("🧠 Clinical reasoning...")
    answer = llm.invoke(f"""
Patient Query: {query}

Evidence:
{summary}

Provide:
- risk
- mechanism
- recommendation
""").content
    
    logs.append("🛡️ Validating...")
    final = llm.invoke(f"Ensure this is safe medical advice:\n{answer}").content
    
    logs.append("✅ Done")
    
    return "\n".join(logs), final

# =========================================
# 🎨 GRADIO UI
# =========================================
def run_app(query):
    logs, result = pharma_live(query)
    return logs, result

demo = gr.Interface(
    fn=run_app,
    inputs=gr.Textbox(label="Enter Clinical Query"),
    outputs=[
        gr.Textbox(label="System Thinking"),
        gr.Textbox(label="Final Clinical Response")
    ],
    title="🧪 Clinical AI Assistant (PubMed Powered)",
    description="Simulates real-world clinical decision system"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [25]:
# =========================================
# 🧪 ULTIMATE CLINICAL AI SYSTEM (MULTI-AGENT + VOICE + PDF + STREAMING)
# =========================================


import gradio as gr
import requests, os
from dotenv import load_dotenv
from pypdf import PdfReader

# load_dotenv("work/.env")

from langchain_groq import ChatGroq
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# =========================================
# 🔬 RESEARCH AGENT (PUBMED)
# =========================================
def pubmed_search(query):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    try:
        res = requests.get(base + "esearch.fcgi",
                           params={"db":"pubmed","term":query,"retmax":2,"retmode":"json"}).json()
        ids = res.get("esearchresult", {}).get("idlist", [])
        
        if not ids:
            return "No strong evidence found."
        
        data = requests.get(base + "efetch.fcgi",
                            params={"db":"pubmed","id":",".join(ids),"retmode":"text","rettype":"abstract"}).text
        
        return data[:1200]
    except:
        return "Error retrieving research"

# =========================================
# 🧠 AGENT 1: RESEARCHER
# =========================================
def researcher_agent(query):
    sub_queries = llm.invoke(f"Generate 2 PubMed search queries for: {query}").content.split("\n")
    
    evidence = ""
    for q in sub_queries[:2]:
        evidence += pubmed_search(q) + "\n\n"
    
    return evidence

# =========================================
# 👨‍⚕️ AGENT 2: DOCTOR
# =========================================
def doctor_agent(query, evidence):
    return llm.invoke(f"""
You are a clinical expert.

Patient Query: {query}

Evidence:
{evidence}

Provide:
- risk
- mechanism
- recommendation
""").content

# =========================================
# 🛡️ AGENT 3: VALIDATOR
# =========================================
def validator_agent(answer):
    return llm.invoke(f"""
Check medical safety.
Fix if needed.

Answer:
{answer}
""").content

# =========================================
# 📄 PDF READER
# =========================================
def extract_pdf(file):
    if file is None:
        return ""
    reader = PdfReader(file.name)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text[:1500]

# =========================================
# 🎤 VOICE INPUT (Gradio handles speech → text)
# =========================================

# =========================================
# 🔄 STREAMING PIPELINE
# =========================================
def full_system(query, pdf):
    context = extract_pdf(pdf)
    
    if context:
        query = query + "\nPatient History:\n" + context
    
    yield "🧠 Researching...", ""
    
    evidence = researcher_agent(query)
    yield "🔬 Evidence Retrieved", ""
    
    answer = doctor_agent(query, evidence)
    yield "👨‍⚕️ Clinical Reasoning", ""
    
    final = validator_agent(answer)
    yield "🛡️ Validated", final

# =========================================
# 🎨 UI
# =========================================
with gr.Blocks() as demo:
    
    gr.Markdown("# 🧪 Clinical AI Copilot (Multi-Agent System)")
    
    with gr.Row():
        query = gr.Textbox(label="Enter Clinical Query")
        audio = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Speak Query (optional)")
    
    pdf = gr.File(label="Upload Patient Report (PDF)")
    
    output_logs = gr.Textbox(label="System Status")
    output_result = gr.Textbox(label="Final Response")
    
    btn = gr.Button("Run Analysis")
    
    btn.click(full_system, inputs=[query, pdf], outputs=[output_logs, output_result])

demo.launch()

/usr/local/lib/python3.10/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
